[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Biswajit1999/daily-astro-notebooks/blob/master/gaia/2026-07-30-parallax-distance-vega/notebook.ipynb)

# How far away is Vega? Measuring it from a real parallax

**Learning goals** — after this notebook you'll be able to:
- Query a real astrometric catalog (Gaia DR3 via `astroquery`, and Hipparcos via Vizier) for a specific named star.
- Recognize when Gaia's own astrometry is unreliable for a target (very bright/saturated stars) and fall back to an appropriate alternative honestly.
- Propagate a parallax measurement error into a distance uncertainty (`sigma_d = sigma_p/p^2`) and compare against an independent literature distance.
- Compute a star's tangential velocity from proper motion and distance.

**Background.** Parallax is the tiny apparent yearly wobble a nearby star shows against the distant background sky as Earth orbits the Sun; a parallax `p` in arcseconds gives a distance `d = 1/p` in parsecs directly, with no other assumptions -- it's the most direct geometric distance measurement in astronomy. Proper motion is how fast a star appears to creep across the sky per year; combined with a distance, it converts into a real tangential velocity in km/s via `v_t = 4.74 * mu[arcsec/yr] * d[pc]`. Vega is bright enough that it saturates Gaia's detectors, so its Gaia DR3 astrometric solution is known to be unreliable -- I check that directly below and use the Hipparcos catalog instead, which was designed for bright naked-eye stars.

## 1. Try Gaia DR3 first, and show why it fails for Vega

In [ ]:
from astroquery.gaia import Gaia
from astroquery.vizier import Vizier
import numpy as np

# Vega coordinates (ICRS, degrees)
ra, dec = 279.234735, 38.783689

query = f'''
SELECT TOP 5 source_id, ra, dec, parallax, parallax_error, pmra, pmdec, phot_g_mean_mag, ruwe
FROM gaiadr3.gaia_source
WHERE 1=CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', {ra}, {dec}, 0.1))
ORDER BY phot_g_mean_mag ASC
'''
job = Gaia.launch_job(query)
gaia_nearby = job.get_results()
print(gaia_nearby)
print()
print("Vega's true apparent magnitude is V = 0.03, far brighter than the faintest Gaia")
print("detection limit but ALSO far brighter than Gaia's bright-star saturation limit (G ~ 3-6 mag,")
print("depending on calibration). None of the sources returned here (all G > 8) is actually")
print("Vega -- Gaia DR3 does not carry a usable direct astrometric solution for a star this")
print("bright at this position; the true star saturates and is dropped from routine processing.")

## 2. Fall back to Hipparcos, the catalog built for bright stars

Hipparcos was an ESA astrometric mission (1989-1993) purpose-built to get precise parallaxes for
~118,000 bright, naked-eye stars -- exactly the regime where Gaia's routine pipeline struggles.
I query the Vizier-hosted van Leeuwen (2007) re-reduction of the Hipparcos catalog (`I/311/hip2`),
which supersedes the original 1997 reduction.

In [ ]:
v = Vizier(columns=['HIP','Plx','e_Plx','pmRA','pmDE','Vmag'])
hip_result = v.query_object('Vega', catalog='I/311/hip2')[0]
print(hip_result)

hip_plx = float(hip_result['Plx'][0])       # mas
hip_plx_err = float(hip_result['e_Plx'][0]) # mas
hip_pmra = float(hip_result['pmRA'][0])     # mas/yr
hip_pmdec = float(hip_result['pmDE'][0])    # mas/yr

print(f"\nHipparcos (van Leeuwen 2007) parallax for Vega (HIP 91262): "
      f"{hip_plx:.2f} +/- {hip_plx_err:.2f} mas")

## 3. Distance, uncertainty propagation, and comparison to the literature

In [ ]:
# distance from parallax; propagate the parallax error into a distance error
# d [pc] = 1000 / p[mas]; sigma_d = sigma_p / p^2 (in matching units, here with the 1000 mas->arcsec scaling folded in)
distance_pc = 1000.0 / hip_plx
distance_err_pc = 1000.0 * hip_plx_err / hip_plx**2

literature_distance_pc = 7.68  # independent literature value for Vega, pc
pct_diff = 100 * (distance_pc - literature_distance_pc) / literature_distance_pc

print(f"Hipparcos-derived distance: {distance_pc:.3f} +/- {distance_err_pc:.3f} pc "
      f"({distance_pc*3.26156:.2f} light-years)")
print(f"Independent literature distance: {literature_distance_pc:.2f} pc")
print(f"Percent difference: {pct_diff:+.2f}%")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5,4))
ax.errorbar([0], [distance_pc], yerr=[distance_err_pc], fmt='o', ms=10, capsize=6, label='Hipparcos (this notebook)')
ax.axhline(literature_distance_pc, color='gray', ls='--', label='literature value')
ax.set_xticks([]); ax.set_ylabel('Distance (pc)')
ax.set_title('Vega: parallax distance vs. literature')
ax.legend()
plt.tight_layout()
plt.savefig('vega_distance_comparison.png', dpi=130)
plt.show()

## 4. Tangential velocity from proper motion + distance

In [ ]:
mu_total_mas = np.hypot(hip_pmra, hip_pmdec)  # total proper motion, mas/yr
mu_total_arcsec = mu_total_mas / 1000.0

# v_t [km/s] = 4.74 * mu[arcsec/yr] * d[pc]
v_tan = 4.74 * mu_total_arcsec * distance_pc
v_tan_err = 4.74 * mu_total_arcsec * distance_err_pc  # dominated by distance uncertainty here

print(f"Total proper motion: {mu_total_mas:.2f} mas/yr")
print(f"Tangential velocity: {v_tan:.2f} +/- {v_tan_err:.2f} km/s")
print(f"(This is only the sky-plane component; the true 3D space velocity also needs the radial velocity.)")

## What I'd look at next

- Add Vega's radial velocity (from spectroscopy) to combine with the tangential velocity here into a full 3D space velocity (U, V, W).
- Check whether the Gaia DR3 *non-single-star* or bright-star-specific reprocessing (rather than the main `gaia_source` table) has a usable solution for Vega.
- Compare this Hipparcos parallax to earlier ground-based parallax measurements to see how much precision improved between eras.

**Citation:** Hipparcos catalog (van Leeuwen 2007 re-reduction, `I/311/hip2`) via VizieR/CDS; Gaia DR3 (`gaiadr3.gaia_source`), ESA Gaia mission. See the Gaia credits page: https://www.cosmos.esa.int/web/gaia-users/credits